# UITI_VANO regression budget — Kaggle run (`uiti-vano-regression` family)

Reproduces `notebooks/project_flow/02.1_mgcecdl_regression_embeddings.ipynb`
on Kaggle with a larger search budget, per `.claude/skills/experimento-kaggle/SKILL.md`'s Experiment
Families dispatch and Dataset Transport section.

**Reuse-first**: `MGCECDLRegressor`, `MGCECDLRegressionLoss`, and
`KernelDensityWeightedMSELoss` are imported from `chec_impacto.models.mgcecdl`
(private Kaggle dataset `chec-impacto-src`) — never redefined here. Data
comes from the private dataset `uiti-vano-indicadores-v3`
(`Indicadores_vano_v3.csv` + `Variables_seleccion.xlsx`).

**Baseline to beat** (pinned in
`.claude/skills/experimento-kaggle/references/uiti-vano-regression-baseline.md`,
local run: `device=mps:0`, loss sweep @20 epochs, Optuna 10 trials @20
epochs, final retrain @60 epochs):

| Metric | Value |
|---|---|
| `mae_original` (primary) | 126.402 |
| `r2_original` | -0.027 |
| `r2_transformed` | 0.284 |
| `ARI` (auto K) | 0.0000 |
| `ARI` (K=4) | 0.1115 |

Methodology order (must be preserved): loss-shape sweep (select by
`mae_original.idxmin()`) -> Optuna (`GPSampler` + `MedianPruner`) -> final
retrain -> embeddings -> K-Means + silhouette `K=2..8` -> ARI vs Part A.

In [ ]:
# Papermill parameters cell. Overridden with `-p mode full` for a real run;
# defaults to the tiny, fast "smoke" mode so a bare execution never
# accidentally launches the full search budget.
mode = "smoke"

## Bootstrap: precondition guard + `sys.path` + device

Precondition-guard mirror required by the skill's Hard Rules and by
`references/uiti-vano-regression-baseline.md`'s "In-Notebook Mirror"
section: this must fail fast, with an actionable message, if the attached
Kaggle dataset does not carry `MGCECDLRegressor`/`MGCECDLRegressionLoss`
(e.g. the wrong dataset version was attached).

Path resolution supports two environments on purpose:
1. **Kaggle**: reads from `/kaggle/input/chec-impacto-src/src` and
   `/kaggle/input/uiti-vano-indicadores-v3/` (the private datasets pinned in
   `SKILL.md`'s Dataset Transport section).
2. **Local smoke-testing** (e.g. `papermill notebook.ipynb out.ipynb -p mode
   smoke`, run from a checkout that already has `MGCECDLRegressor` merged):
   falls back to the local `src/`/`data/` next to the checkout, exactly like
   `resolve_project_root()` in the local notebook this reproduces.

In [ ]:
import sys
from pathlib import Path

KAGGLE_SRC_DATASET = "chec-impacto-src"
KAGGLE_DATA_DATASET = "uiti-vano-indicadores-v3"


def resolve_src_dir() -> Path:
    kaggle_src = Path(f"/kaggle/input/{KAGGLE_SRC_DATASET}/src")
    if kaggle_src.exists():
        return kaggle_src
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate / "src"
    raise FileNotFoundError(
        "Could not resolve src/chec_impacto/: neither "
        f"/kaggle/input/{KAGGLE_SRC_DATASET}/src nor a local checkout's "
        "src/chec_impacto/ was found."
    )


def resolve_data_dir() -> Path:
    kaggle_data = Path(f"/kaggle/input/{KAGGLE_DATA_DATASET}")
    if kaggle_data.exists():
        return kaggle_data
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "Indicadores_vano_v3.csv").exists():
            return candidate / "data"
    raise FileNotFoundError(
        "Could not resolve the data directory: neither "
        f"/kaggle/input/{KAGGLE_DATA_DATASET} nor a local checkout's data/ "
        "was found."
    )


SRC_DIR = resolve_src_dir()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
DATA_DIR = resolve_data_dir()

# Kaggle mounts /kaggle/input read-only; scratch space (Optuna journals,
# figures) must live under /kaggle/working when running there.
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else SRC_DIR.parent
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)

# --- Precondition guard: import probe, never a substitute implementation ---
try:
    from chec_impacto.models.mgcecdl import (
        MGCECDLRegressor,
        MGCECDLRegressionLoss,
        KernelDensityWeightedMSELoss,
    )
except ImportError as exc:
    raise SystemExit(
        "MGCECDLRegressor/MGCECDLRegressionLoss not importable -- the attached "
        f"src dataset ({KAGGLE_SRC_DATASET}) appears to be missing or out of "
        "date. These classes live in src/chec_impacto/models/mgcecdl.py on "
        f"main; re-package/version the {KAGGLE_SRC_DATASET} Kaggle dataset "
        "from the current main checkout and re-attach it to this kernel."
    ) from exc

print("Guard passed: MGCECDLRegressor/MGCECDLRegressionLoss/KernelDensityWeightedMSELoss importable.")

In [ ]:
import json
import time as _time

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, mean_absolute_error, r2_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from chec_impacto.data import construir_matriz_adyacencia_mgcecdl, procesar_dataset_completo
from chec_impacto.training import (
    calcular_estadisticas_reconstruccion_mgcecdl,
    construir_modalidades_mgcecdl,
    guardar_estudio_optuna,
    resolve_training_device,
)
from chec_impacto.training.mgcecdl import run_optuna_study
import optuna

RANDOM_STATE = 42
DEVICE = resolve_training_device("auto")
print(f"Using device: {DEVICE}")

## Mode configuration (`smoke` vs `full`)

In [ ]:
if mode not in ("smoke", "full"):
    raise ValueError(f"Unknown mode {mode!r} -- expected 'smoke' or 'full'.")

if mode == "smoke":
    # Tiny budget: few epochs/trials, small data subsample -- must finish
    # well under a minute (SKILL.md Hard Rule).
    SMOKE_EVENT_SUBSAMPLE = 2000
    LOSS_COMPARISON_MAX_EPOCHS = 2
    LOSS_COMPARISON_PATIENCE = 2
    OPTUNA_N_TRIALS = 2
    OPTUNA_MAX_EPOCHS = 2
    OPTUNA_PATIENCE = 2
    FINAL_MAX_EPOCHS = 3
    FINAL_PATIENCE = 3
else:
    # `full` mode: budget strictly >= the local MPS baseline (10 trials @20
    # epochs sweep/search, 60-epoch final retrain) -- same code, same metric
    # definitions, only the search budget grows (SKILL.md Dataset
    # Transport / Kaggle Budget Delta note).
    SMOKE_EVENT_SUBSAMPLE = None
    LOSS_COMPARISON_MAX_EPOCHS = 20
    LOSS_COMPARISON_PATIENCE = 7
    OPTUNA_N_TRIALS = 15  # > 10 (baseline), per spec's Increased Kaggle Search Budget requirement
    OPTUNA_MAX_EPOCHS = 20  # not reduced vs baseline
    OPTUNA_PATIENCE = 7
    FINAL_MAX_EPOCHS = 60  # not reduced vs baseline
    FINAL_PATIENCE = 15

print(f"mode={mode!r}")
print(
    f"loss_sweep_epochs={LOSS_COMPARISON_MAX_EPOCHS} | optuna_trials={OPTUNA_N_TRIALS} | "
    f"optuna_epochs={OPTUNA_MAX_EPOCHS} | final_epochs={FINAL_MAX_EPOCHS} | "
    f"event_subsample={SMOKE_EVENT_SUBSAMPLE}"
)

## 1. Data loading

Same call as the local notebook: `procesar_dataset_completo` (log1p +
MinMax target transform, 12h climate window, no UITI cap -- decisions
already justified and fixed in the local baseline, not re-derived here).
The adjacency graph is always rebuilt in-process (`construir_matriz_adyacencia_mgcecdl`)
rather than read from an on-disk cache: `data/graphs/` is an **optional**
rebuild cache locally and is not part of the `uiti-vano-indicadores-v3`
dataset (see `SKILL.md`'s Dataset Transport table) -- omitting it here is a
documented, deliberate choice, not a regression.

In [ ]:
VENTANA_CLIMATICA_HORAS = 12
FILTRO_UITI_MAX = None

DATASET_PATH = DATA_DIR / "Indicadores_vano_v3.csv"
VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing {DATASET_PATH} -- check the attached data dataset.")
if not VARIABLES_SELECCION_PATH.exists():
    raise FileNotFoundError(f"Missing {VARIABLES_SELECCION_PATH} -- check the attached data dataset.")

datos_procesados = procesar_dataset_completo(
    path_clima=DATASET_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target="UITI_VANO",
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

X = datos_procesados["X"]
y = datos_procesados["y"]
features = datos_procesados["features"]
df_identidad = datos_procesados["df_original_copy"].reset_index(drop=True)
assert len(df_identidad) == len(X) == len(y)

if SMOKE_EVENT_SUBSAMPLE is not None and len(df_identidad) > SMOKE_EVENT_SUBSAMPLE:
    rng = np.random.RandomState(RANDOM_STATE)
    subsample_idx = np.sort(rng.choice(len(df_identidad), size=SMOKE_EVENT_SUBSAMPLE, replace=False))
    X = X[subsample_idx]
    y = y[subsample_idx]
    df_identidad = df_identidad.iloc[subsample_idx].reset_index(drop=True)

modality_feature_indices = construir_modalidades_mgcecdl(features)
graph_adjacency_matrix, _edges = construir_matriz_adyacencia_mgcecdl(
    features, ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)
assert graph_adjacency_matrix.shape == (len(features), len(features))

print("X shape:", X.shape, "| y shape:", y.shape)
print("Modalities:", {name: len(indices) for name, indices in modality_feature_indices.items()})
print("Adjacency matrix:", graph_adjacency_matrix.shape)

## 2. Chronological train/valid split (percentile-80 date cut, not random)

In [ ]:
fechas = df_identidad["FECHA"]
FECHA_CORTE = fechas.quantile(0.8)
train_mask = (fechas <= FECHA_CORTE).to_numpy()
valid_mask = ~train_mask

print(f"Fecha de corte (percentil 80): {FECHA_CORTE}")
print(f"Train: {train_mask.sum()} eventos | Valid: {valid_mask.sum()} eventos")

## 3. Scaling, tensors, and graph-reconstruction statistics

In [ ]:
x_scaler = MinMaxScaler()
X_train_scaled = x_scaler.fit_transform(X[train_mask]).astype(np.float32)
X_valid_scaled = x_scaler.transform(X[valid_mask]).astype(np.float32)
X_full_scaled = x_scaler.transform(X).astype(np.float32)

y_log1p_all = np.log1p(y)
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_log1p_all[train_mask]).astype(np.float32).reshape(-1)
y_valid_scaled = y_scaler.transform(y_log1p_all[valid_mask]).astype(np.float32).reshape(-1)

feature_mean, feature_std = calcular_estadisticas_reconstruccion_mgcecdl(X_train_scaled)

BATCH_SIZE = 512
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def make_loaders(batch_size=BATCH_SIZE, seed=RANDOM_STATE):
    train_dataset = TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train_scaled))
    valid_dataset = TensorDataset(torch.tensor(X_valid_scaled), torch.tensor(y_valid_scaled))
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    valid_loader = DataLoader(valid_dataset, batch_size=1024, shuffle=False)
    return train_loader, valid_loader


train_loader, valid_loader = make_loaders()
print("Train batches:", len(train_loader), "| Valid batches:", len(valid_loader))

## 4. Shared training loop (`train_regressor`)

Byte-identical logic to the local notebook's `train_regressor`: same loss
composition call (`loss_fn.compute_components`), same device-landing assert
(fails loudly instead of silently training on CPU when a GPU/MPS device was
requested), same best-checkpoint-by-`valid_fused_loss` selection, same
metric set (`mae_original`, `r2_original`, `r2_transformed`).

In [ ]:
def build_regression_loss(base_loss, gamma_sup=0.20, gamma_agr=0.10, gamma_reg=0.01, kernel_loss_module=None,
                           lambda_reconstruction=0.01, lambda_mutual_information=0.01):
    return MGCECDLRegressionLoss(
        base_loss=base_loss,
        gamma_sup=gamma_sup,
        gamma_agr=gamma_agr,
        gamma_reg=gamma_reg,
        kernel_loss_module=kernel_loss_module,
        feature_mean=feature_mean,
        feature_std=feature_std,
        adjacency_matrix=graph_adjacency_matrix,
        rbf_sigma=1.0,
        lambda_reconstruction=lambda_reconstruction,
        lambda_mutual_information=lambda_mutual_information,
    )


def train_regressor(
    loss_fn,
    hidden_dim=128,
    embed_dim=64,
    dropout=0.10,
    lr=1e-3,
    weight_decay=1e-5,
    optimizer_type="adamw",
    momentum=0.0,
    batch_size=BATCH_SIZE,
    max_epochs=60,
    patience=15,
    seed=RANDOM_STATE,
    verbose=False,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    local_train_loader, local_valid_loader = make_loaders(batch_size=batch_size, seed=seed)

    model = MGCECDLRegressor(
        modality_feature_indices=modality_feature_indices,
        hidden_dim=hidden_dim, embed_dim=embed_dim, dropout=dropout,
    ).to(DEVICE)
    loss_fn = loss_fn.to(DEVICE)
    actual_param_device = next(model.parameters()).device
    assert actual_param_device.type == torch.device(DEVICE).type, (
        f"Model landed on {actual_param_device} but DEVICE={DEVICE} was requested -- "
        "silent fallback, aborting instead of training on the wrong device."
    )

    optimizer_type = optimizer_type.lower()
    if optimizer_type == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_type == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_type == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_type == "rmsprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_type}")

    def run_epoch(loader, train):
        model.train(mode=train)
        total_fused = 0.0
        preds, targets_list = [], []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.set_grad_enabled(train):
                out = model(xb)
                components = loss_fn.compute_components(out, yb, xb)
                if train:
                    components["total_loss"].backward()
                    optimizer.step()
            total_fused += float(components["fused_loss"].detach().cpu())
            preds.append(out["fused_prediction"].detach().cpu().numpy())
            targets_list.append(yb.detach().cpu().numpy())
        return total_fused / max(len(loader), 1), np.concatenate(preds), np.concatenate(targets_list)

    history = []
    best_valid_fused = float("inf")
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        _epoch_t0 = _time.time()
        train_fused, _, _ = run_epoch(local_train_loader, train=True)
        valid_fused, _, _ = run_epoch(local_valid_loader, train=False)
        _epoch_elapsed = _time.time() - _epoch_t0
        history.append({"epoch": epoch, "train_fused_loss": train_fused, "valid_fused_loss": valid_fused,
                         "epoch_seconds": _epoch_elapsed})

        if valid_fused < best_valid_fused - 1e-7:
            best_valid_fused = valid_fused
            best_epoch = epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"epoch {epoch:03d} | device={actual_param_device} | epoch_time={_epoch_elapsed:.2f}s | "
                  f"train_fused={train_fused:.5f} | valid_fused={valid_fused:.5f} | best={best_valid_fused:.5f}@{best_epoch}")

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(best_state)
    model.eval()
    _, valid_preds_scaled, valid_targets_scaled = run_epoch(local_valid_loader, train=False)

    valid_preds_log1p = y_scaler.inverse_transform(valid_preds_scaled.reshape(-1, 1)).reshape(-1)
    valid_targets_log1p = y_scaler.inverse_transform(valid_targets_scaled.reshape(-1, 1)).reshape(-1)
    valid_preds_original = np.expm1(valid_preds_log1p)
    valid_targets_original = np.expm1(valid_targets_log1p)

    metrics = {
        "best_epoch": best_epoch,
        "best_valid_fused_loss": best_valid_fused,
        "device": str(actual_param_device),
        "r2_transformed": r2_score(valid_targets_scaled, valid_preds_scaled),
        "r2_original": r2_score(valid_targets_original, valid_preds_original),
        "mae_original": mean_absolute_error(valid_targets_original, valid_preds_original),
    }
    return model, history, metrics

## 5. Loss-shape sweep (`mse` / `huber` / `kernel_weighted_mse`)

Same architecture (`hidden_dim=128, embed_dim=64, dropout=0.10`), same
epoch budget across the three candidates, selection by
`mae_original.idxmin()` -- exactly the local baseline's rule (spec's
"Loss-shape sweep selects by MAE" scenario).

In [ ]:
kernel_loss_module = KernelDensityWeightedMSELoss.from_targets(y_train_scaled, n_grid=512)

loss_variants = {
    "mse": build_regression_loss("mse"),
    "huber": build_regression_loss("huber"),
    "kernel_weighted_mse": build_regression_loss("kernel_weighted_mse", kernel_loss_module=kernel_loss_module),
}

loss_comparison_rows = []
for name, loss_fn in loss_variants.items():
    print(f"--- Training with base_loss={name} ---")
    _, _, metrics = train_regressor(
        loss_fn, max_epochs=LOSS_COMPARISON_MAX_EPOCHS, patience=LOSS_COMPARISON_PATIENCE, verbose=True,
    )
    loss_comparison_rows.append({"base_loss": name, **metrics})

loss_comparison_df = pd.DataFrame(loss_comparison_rows).set_index("base_loss")
BEST_BASE_LOSS = loss_comparison_df["mae_original"].idxmin()
print(loss_comparison_df)
print(f"\nBest base_loss by mae_original: {BEST_BASE_LOSS}")

## 6. Optuna hyperparameter search

Same sampler/pruner as the local baseline (`GPSampler` + `MedianPruner`,
reused via `run_optuna_study` -- never reimplemented), objective is
`mae_original` (minimize). `full` mode's trial/epoch budget is set above to
exceed the local baseline (>10 trials, epochs not reduced); `smoke` mode
just proves the search wiring works end to end.

In [ ]:
OPTUNA_DIR = WORK_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)


def regression_objective(trial):
    params = {
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 192]),
        "embed_dim": trial.suggest_categorical("embed_dim", [32, 64, 96]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.25),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True),
        "optimizer_type": trial.suggest_categorical("optimizer_type", ["adam", "adamw", "sgd", "rmsprop"]),
        "momentum": trial.suggest_float("momentum", 0.5, 0.95),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "gamma_sup": trial.suggest_float("gamma_sup", 1e-2, 1.0, log=True),
        "gamma_agr": trial.suggest_float("gamma_agr", 1e-2, 1.0, log=True),
        "gamma_reg": trial.suggest_float("gamma_reg", 1e-2, 1.0, log=True),
        "lambda_reconstruction": trial.suggest_float("lambda_reconstruction", 1e-2, 1.0, log=True),
        "lambda_mutual_information": trial.suggest_float("lambda_mutual_information", 1e-2, 1.0, log=True),
    }
    loss_fn = build_regression_loss(
        BEST_BASE_LOSS,
        gamma_sup=params["gamma_sup"], gamma_agr=params["gamma_agr"], gamma_reg=params["gamma_reg"],
        kernel_loss_module=kernel_loss_module if BEST_BASE_LOSS == "kernel_weighted_mse" else None,
        lambda_reconstruction=params["lambda_reconstruction"],
        lambda_mutual_information=params["lambda_mutual_information"],
    )
    _, _, metrics = train_regressor(
        loss_fn,
        hidden_dim=params["hidden_dim"], embed_dim=params["embed_dim"], dropout=params["dropout"],
        lr=params["learning_rate"], weight_decay=params["weight_decay"],
        optimizer_type=params["optimizer_type"], momentum=params["momentum"], batch_size=params["batch_size"],
        max_epochs=OPTUNA_MAX_EPOCHS, patience=OPTUNA_PATIENCE,
    )
    return metrics["mae_original"]


_search_t0 = _time.time()
regression_study = run_optuna_study(
    objective=regression_objective,
    study_name=f"mgcecdl_regression_kaggle_budget_{mode}",
    storage_path=OPTUNA_DIR / f"mgcecdl_regression_search_{mode}.journal",
    n_trials=OPTUNA_N_TRIALS,
    seed=RANDOM_STATE,
    direction="minimize",
)
_search_elapsed = _time.time() - _search_t0
guardar_estudio_optuna(regression_study, OPTUNA_DIR / f"mgcecdl_regression_study_{mode}.pkl")

print(f"Optuna search: {OPTUNA_N_TRIALS} trials in {_search_elapsed:.1f}s "
      f"(avg {_search_elapsed / max(OPTUNA_N_TRIALS, 1):.1f}s/trial)")
print(f"Best mae_original: {regression_study.best_value:.4f}")
print(f"Best hyperparameters: {json.dumps(regression_study.best_params, indent=2)}")

## 7. Final retrain (best loss shape + Optuna's best hyperparameters)

In [ ]:
best_params = regression_study.best_params
final_loss_fn = build_regression_loss(
    BEST_BASE_LOSS,
    gamma_sup=best_params["gamma_sup"], gamma_agr=best_params["gamma_agr"], gamma_reg=best_params["gamma_reg"],
    kernel_loss_module=kernel_loss_module if BEST_BASE_LOSS == "kernel_weighted_mse" else None,
    lambda_reconstruction=best_params["lambda_reconstruction"],
    lambda_mutual_information=best_params["lambda_mutual_information"],
)

regressor, final_history, final_metrics = train_regressor(
    final_loss_fn,
    hidden_dim=best_params["hidden_dim"], embed_dim=best_params["embed_dim"], dropout=best_params["dropout"],
    lr=best_params["learning_rate"], weight_decay=best_params["weight_decay"],
    optimizer_type=best_params["optimizer_type"], momentum=best_params["momentum"], batch_size=best_params["batch_size"],
    max_epochs=FINAL_MAX_EPOCHS, patience=FINAL_PATIENCE, verbose=True,
)

print("\n===== Final model (best hyperparameters + best loss shape) =====")
for key, value in final_metrics.items():
    print(f"{key}: {value}")

## 8. Embedding extraction (all events, final model)

In [ ]:
regressor.eval()
full_dataset = TensorDataset(torch.tensor(X_full_scaled))
full_loader = DataLoader(full_dataset, batch_size=2048, shuffle=False)

event_embeddings_chunks = []
with torch.no_grad():
    for (xb,) in full_loader:
        xb = xb.to(DEVICE)
        out = regressor(xb)
        concatenated = torch.cat(out["embeddings"], dim=1)
        event_embeddings_chunks.append(concatenated.cpu().numpy())
event_embeddings = np.vstack(event_embeddings_chunks)

embedding_columns = [f"embed_{i}" for i in range(event_embeddings.shape[1])]
event_embeddings_df = pd.DataFrame(event_embeddings, columns=embedding_columns)
event_embeddings_df["CIRCUITO"] = df_identidad["CIRCUITO"].values
event_embeddings_df["FID_VANO"] = df_identidad["FID_VANO"].astype(str).values

vano_embeddings_df = (
    event_embeddings_df.groupby(["CIRCUITO", "FID_VANO"])[embedding_columns].mean().reset_index()
)
print("event_embeddings shape:", event_embeddings.shape)
print("vano_embeddings_df shape:", vano_embeddings_df.shape)

## 9. K-Means + silhouette validation on the embeddings (`K=2..8`)

In [ ]:
embedding_matrix = StandardScaler().fit_transform(vano_embeddings_df[embedding_columns].values)

n_vanos = embedding_matrix.shape[0]
k_upper_bound = min(9, n_vanos)  # KMeans needs n_samples >= n_clusters; guards tiny smoke subsamples
K_RANGE = range(2, k_upper_bound)
if len(K_RANGE) == 0:
    raise RuntimeError(
        f"Only {n_vanos} distinct vanos in this run -- increase SMOKE_EVENT_SUBSAMPLE "
        "or use mode='full'."
    )

embedding_cluster_rows = []
embedding_labels_by_k = {}
for k in K_RANGE:
    kmeans_embed = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_embed.fit_predict(embedding_matrix)
    silhouette = silhouette_score(embedding_matrix, labels)
    embedding_cluster_rows.append({"k": k, "inertia": kmeans_embed.inertia_, "silhouette": silhouette})
    embedding_labels_by_k[k] = labels

embedding_cluster_df = pd.DataFrame(embedding_cluster_rows).set_index("k")
best_k_embeddings = int(embedding_cluster_df["silhouette"].idxmax())
print(embedding_cluster_df)
print(f"K recommended by silhouette (global max): {best_k_embeddings}")

## 10. Triangulation vs. Part A (raw-feature K-Means from `10_uiti_vano_kmeans.ipynb`)

Recomputed independently from the CSV (same event-level source as the
model input), then joined with the new embeddings by `CIRCUITO` +
`FID_VANO`. `K=4` is triangulated against Part A regardless of the
silhouette-selected `K` above, to stay comparable with the pinned
`ARI(K=4)=0.1115` baseline.

In [ ]:
raw_events_df = pd.read_csv(DATASET_PATH, usecols=["CIRCUITO", "FID_VANO", "UITI_VANO"])
raw_events_df["FID_VANO"] = (
    raw_events_df["FID_VANO"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
)
raw_events_df["UITI_VANO"] = pd.to_numeric(raw_events_df["UITI_VANO"], errors="coerce").fillna(0.0)

vano_table_partA = (
    raw_events_df.groupby(["CIRCUITO", "FID_VANO"])
    .agg(uiti_acumulado=("UITI_VANO", "sum"), num_eventos=("UITI_VANO", "count"))
    .reset_index()
)
log_features_partA = np.column_stack([
    np.log10(vano_table_partA["num_eventos"]), np.log10(vano_table_partA["uiti_acumulado"]),
])
log_features_partA_scaled = MinMaxScaler().fit_transform(log_features_partA)

PART_A_SILHOUETTE_K = 2
PART_A_PRODUCTION_K = 4
vano_table_partA["cluster_partA_silhouette_k"] = KMeans(
    n_clusters=PART_A_SILHOUETTE_K, random_state=42, n_init=10
).fit_predict(log_features_partA_scaled)
vano_table_partA["cluster_partA_k4"] = KMeans(
    n_clusters=PART_A_PRODUCTION_K, random_state=42, n_init=10
).fit_predict(log_features_partA_scaled)

vano_embeddings_df["cluster_partB_silhouette_k"] = embedding_labels_by_k[best_k_embeddings]
vano_embeddings_df["cluster_partB_k4"] = embedding_labels_by_k.get(4, embedding_labels_by_k[best_k_embeddings])

triangulation_df = vano_table_partA.merge(
    vano_embeddings_df[["CIRCUITO", "FID_VANO", "cluster_partB_silhouette_k", "cluster_partB_k4"]],
    on=["CIRCUITO", "FID_VANO"], how="inner",
)
print("vanos in Part A:", len(vano_table_partA))
print("vanos in Part B (this run's embeddings):", len(vano_embeddings_df))
print("vanos with both groupings (join):", len(triangulation_df))

ari_own_best_k = adjusted_rand_score(
    triangulation_df["cluster_partA_silhouette_k"], triangulation_df["cluster_partB_silhouette_k"]
)
ari_matched_k4 = adjusted_rand_score(triangulation_df["cluster_partA_k4"], triangulation_df["cluster_partB_k4"])
print(f"ARI (own K per side -- Part A K={PART_A_SILHOUETTE_K}, Part B K={best_k_embeddings}): {ari_own_best_k:.4f}")
print(f"ARI (K=4 fixed both sides): {ari_matched_k4:.4f}")

## 11. Report: `mae_original` vs. the pinned local baseline

`mae_original` is the primary success/comparison metric (spec's "MAE Is
the Reported Success/Comparison Metric" requirement); `r2_original`/
`r2_transformed`/ARI are secondary diagnostics and never replace it. The
baseline constant below is pinned in
`.claude/skills/experimento-kaggle/references/uiti-vano-regression-baseline.md`
(that file is not part of any Kaggle dataset, so it is not readable from a
Kaggle kernel at runtime -- the value is copied here deliberately, with a
pointer back to the source of truth, rather than read live).

In [ ]:
BASELINE_MAE_ORIGINAL = 126.402  # pinned in references/uiti-vano-regression-baseline.md
BASELINE_R2_ORIGINAL = -0.027
BASELINE_R2_TRANSFORMED = 0.284
BASELINE_ARI_AUTO_K = 0.0000
BASELINE_ARI_K4 = 0.1115

run_mae_original = final_metrics["mae_original"]
delta = run_mae_original - BASELINE_MAE_ORIGINAL
improved = run_mae_original < BASELINE_MAE_ORIGINAL

print(f"mode: {mode}")
print(f"run mae_original:      {run_mae_original:.4f}")
print(f"baseline mae_original: {BASELINE_MAE_ORIGINAL:.4f}")
print(f"delta (run - baseline): {delta:+.4f}  ({'IMPROVED (lower)' if improved else 'did not improve'})")
print()
print("Secondary diagnostics (this run vs. baseline):")
print(f"  r2_original:      {final_metrics['r2_original']:.4f}  vs. {BASELINE_R2_ORIGINAL:.4f}")
print(f"  r2_transformed:   {final_metrics['r2_transformed']:.4f}  vs. {BASELINE_R2_TRANSFORMED:.4f}")
print(f"  ARI (auto K):     {ari_own_best_k:.4f}  vs. {BASELINE_ARI_AUTO_K:.4f}")
print(f"  ARI (K=4):        {ari_matched_k4:.4f}  vs. {BASELINE_ARI_K4:.4f}")
print()
print(f"Optuna: {OPTUNA_N_TRIALS} trials @ {OPTUNA_MAX_EPOCHS} epochs | "
      f"loss shape selected: {BEST_BASE_LOSS} | final retrain: {FINAL_MAX_EPOCHS} epochs | device: {DEVICE}")